# DCF Valuation Demo

Builds a FCFF DCF from three-statement outputs, values the business under both perpetual growth and exit-multiple methods, and runs WACC × Growth / WACC × Multiple sensitivities plus Bear/Base/Bull scenarios.

In [ ]:
from src.financial_model.assumptions import ModelAssumptions, BEAR_SCENARIO, BASE_SCENARIO, BULL_SCENARIO
from src.financial_model.three_statement import ThreeStatementModel
from src.valuation.wacc import calculate_wacc, calculate_cost_of_equity
from src.valuation.dcf import DCFModel, DCFInputs
from src.valuation.sensitivity import sensitivity_wacc_growth, sensitivity_wacc_exit_multiple, scenario_comparison
from src.visualization.charts import plot_fcff_forecast, plot_sensitivity_heatmap
import pandas as pd

# Base forecast
assumptions = ModelAssumptions(
    historical_revenue=500,
    revenue_growth=[0.06, 0.05, 0.04, 0.03, 0.03],
    gross_margin=0.40, opex_margin=0.20, da_pct_revenue=0.03, capex_pct_revenue=0.04,
    tax_rate=0.25, ar_days=45, inventory_days=30, ap_days=40,
    beginning_cash=80, beginning_ar=62, beginning_inventory=25, beginning_ppe=200,
    beginning_other_assets=20, beginning_ap=30, beginning_debt=150, beginning_other_liabilities=10,
    beginning_equity=197, beginning_retained_earnings=100,
    interest_rate=0.05, dividends_pct_net_income=0.20, share_count=10,
    forecast_years=5,
)
model = ThreeStatementModel(assumptions)
model.income_statement.round(1)

In [ ]:
# WACC via CAPM
ke = calculate_cost_of_equity(risk_free_rate=0.03, beta=1.1, equity_risk_premium=0.05)
wacc = calculate_wacc(market_cap=800, debt=150, cost_of_equity=ke, cost_of_debt=0.05, tax_rate=0.25)
print(f"Ke={ke:.2%}, WACC={wacc:.2%}")

In [ ]:
# DCF — direct from three-statement
dcf = DCFModel.from_three_statement(model, wacc=wacc, terminal_growth=0.02, exit_multiple=10)
display(dcf.build().round(1))
print("\nPerpetual Growth:", dcf.summary("growth"))
print("\nExit Multiple:", dcf.summary("multiple"))

In [ ]:
# FCFF chart
plot_fcff_forecast(dcf.to_dataframe(), save_path="../outputs/fcff_forecast.png")

In [ ]:
# Sensitivity: WACC x Terminal Growth
# Need raw DCFInputs for sensitivity grids
from src.valuation.dcf import DCFInputs
inputs = dcf.inputs  # constructed from three-statement
wacc_range = [0.07, 0.08, 0.09, 0.10, 0.11]
growth_range = [0.01, 0.015, 0.02, 0.025, 0.03]
sens_g = sensitivity_wacc_growth(inputs, wacc_range, growth_range)
sens_g.round(1)


In [ ]:
plot_sensitivity_heatmap(sens_g, title="WACC × Terminal Growth → Implied Price", save_path="../outputs/sensitivity_growth.png", base_wacc="9.0%", base_growth="2.0%")

In [ ]:
# Sensitivity: WACC x Exit Multiple
mult_range = [8, 9, 10, 11, 12]
sens_m = sensitivity_wacc_exit_multiple(inputs, wacc_range, mult_range)
sens_m.round(1)


In [ ]:
plot_sensitivity_heatmap(sens_m, title="WACC × Exit Multiple → Implied Price", save_path="../outputs/sensitivity_multiple.png")

In [ ]:
# Scenario comparison
scen_df = scenario_comparison(
    assumptions,
    scenarios=[BEAR_SCENARIO, BASE_SCENARIO, BULL_SCENARIO],
    wacc_map={"Bear": 0.10, "Base": 0.09, "Bull": 0.08},
    terminal_growth_map={"Bear": 0.01, "Base": 0.02, "Bull": 0.025},
    current_price=80,
)
scen_df.round(2)